In [3]:
from pathlib import Path
from typing import List
import re

def print_file_content(file_path: Path):
    content = file_path.read_text()

    # Remove single-line comments
    content = re.sub(r"//.*", "", content)

    # Remove multi-line comments
    content = re.sub(r"/\*[\s\S]*?\*/", "", content)

    print(f"\n\nFile: '{file_path}'")
    print("\n", content, "")

def process_files(file_patterns: List[str]):
    for pattern in file_patterns:
        for file in Path().glob(pattern):
            print_file_content(file)

file_patterns = ["../../src/*.ts"]
process_files(file_patterns)



File: '../../src/streamUtils.ts'

 import axios, { AxiosResponse } from "axios";
import { CreateChatCompletionResponse } from "openai";
import { CancellationToken, window } from "vscode";
import { FinishReason } from "./finishReason";

export async function* streamChatCompletion(
  response: AxiosResponse<CreateChatCompletionResponse, AsyncIterable<Buffer>>,
  token: CancellationToken
): AsyncGenerator<string | FinishReason, void, undefined> {
  
  const dataStream = response.data as unknown as AsyncIterable<Buffer>;

  for await (const chunk of dataStream) {
    if (token.isCancellationRequested) {
      throw new axios.Cancel("ChatCompletion API request cancelled by user");
    }

    const lines = chunk
      .toString("utf8")
      .split("\n")
      .filter((line) => line.trim().startsWith("data: "));

    for (const line of lines) {
      const message = line.replace(/^data: /, "");
      const json = JSON.parse(message);

      const content = json.choices[0].delta.content;

 

**The VSCode extension above only support Python Jupyter Notebooks. I want the extension to also support Polyglot Notebooks (C#, F#, PowerShell, JavaScript, SQL, KQL, HTML, Mermaid). Please outline the changes necessary to support Polyglot Notebook and Python Notebooks at the same time.**
**Ideally, the detection of code cells relies on the languageIds supported by the current kernel instead of being hardcoded. However, I don't know if it's possible to retrieve the languageIds supported by the current kernel.**

To support Polyglot Notebooks and Python Notebooks at the same time, you need to make the following changes:

1. Detect the language of the code cell based on the `languageId` property of the cell document. This will allow the extension to work with multiple languages without hardcoding them.
 

**STOP - that won't work because the extension creates the code cell. Currently, it receives text from the API and when it detects a python code block, it makes a code cell, otherwise it makes a markdown cell. To reliably detect if I can create a code cell in a specific language, I have to compare the code block's language identifier with the languageIds supported by the current kernel.**

I understand your concern. In that case, you can follow these steps to support Polyglot Notebooks and Python Notebooks:

1. Retrieve the list of supported languageIds by the current kernel. You can use the `notebook.kernel` property of the active notebook editor to get the kernel information. Unfortunately, the VSCode API does not provide a direct way to get the supported languageIds from the kernel. As a workaround, you can use the kernel's `id` or `label` to map it to the supported languageIds. You can create a mapping of kernel ids/labels to their supported languageIds in your extension.

2. When you receive text from the API, parse the code block's language identifier. You can use a regular expression to extract the language identifier from the code block, e.g., for a code block like "```python", extract "python" as the language identifier.

3. Compare the extracted language identifier with the list of supported languageIds by the current kernel. If the language identifier is in the list of supported languageIds, create a code cell with that languageId. Otherwise, create a markdown cell.

Here's a high-level outline of the changes:

1. Create a mapping of kernel ids/labels to their supported languageIds.

```typescript
const kernelLanguageMapping = {
  "python": ["python"],
  "csharp": ["csharp"],
  "fsharp": ["fsharp"],
  "powershell": ["powershell"],
  "javascript": ["javascript"],
  "sql": ["sql"],
  "kql": ["kql"],
  "html": ["html"],
  "mermaid": ["mermaid"],
};
```

2. Extract the language identifier from the code block.

```typescript
const codeBlockRegex = /```(\w+)\n/;
const match = codeBlockRegex.exec(textToken);
const languageIdentifier = match ? match[1] : null;
```

3. Check if the language identifier is supported by the current kernel.

```typescript
const kernel = window.activeNotebookEditor.notebook.kernel;
const supportedLanguageIds = kernelLanguageMapping[kernel.id] || kernelLanguageMapping[kernel.label];

if (languageIdentifier && supportedLanguageIds.includes(languageIdentifier)) {
  // Create a code cell with the languageId
  ck = NotebookCellKind.Code;
  cellIndex = await insertCell(editor, cellIndex, ck, languageIdentifier);
  textToken = textToken.replace(`\`\`\`${languageIdentifier}\n`, "");
} else {
  // Create a markdown cell
}
```

By following these steps, your extension should be able to support both Polyglot Notebooks and Python Notebooks.